In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1nyKplyA0n97BQBFGe76HPZkHBwGA83eR", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Agent SDK Hooks & Enforcement — Hands-On Lab

Welcome to your third hands-on lab for Domain 1 of the Claude Certified Architect exam. In this notebook, we will **build a hook-based enforcement system from scratch** — the same PostToolUse and PreToolUse pattern the Agent SDK uses to guarantee compliance without relying on prompts.

By the end of this notebook, you will be able to:
- Implement PostToolUse hooks that normalize tool results before the model sees them
- Build PreToolUse hooks that intercept and block policy-violating tool calls
- Explain why hooks provide deterministic guarantees that prompts cannot
- Wire multiple hooks into a mock Agent SDK class

**Estimated time: 45–55 minutes**

## AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/agentic-architecture/practice/3/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you are building a financial operations agent. It can look up accounts, process refunds, and modify billing settings. Your company has a hard rule: no refund over $500 without a manager's approval. You could tell the model in the system prompt: "Never process refunds over $500." But what happens when a customer says "I was charged $750 by mistake and I need it back urgently"? The model might sympathize and process it anyway.

This is not hypothetical — **prompts are probabilistic, hooks are deterministic.** The exam dedicates significant coverage to understanding when to use each. PostToolUse hooks normalize data formats so the model always sees clean inputs. PreToolUse hooks block forbidden actions with 100% reliability. Understanding this distinction is essential for the 27% of the exam covering agentic architecture.

Let us build both from scratch.

In [ ]:
#@title 🎧 Listen: Prompts Vs Hooks
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_prompts_vs_hooks.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — Prompts vs Hooks

Think of a prompt-based rule like a speed limit sign on the highway. Most drivers obey it, but some will exceed it — especially when they feel justified. A hook is like a physical speed bump. No matter how fast you want to go, the bump enforces the limit mechanically.

In [ ]:
# Setup
import json
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional
from datetime import datetime, timezone

# Prompt-based "enforcement" — the sign on the highway
SYSTEM_PROMPT_APPROACH = """
You MUST NOT process refunds over $500.
You MUST always convert timestamps to ISO 8601.
"""

# Hook-based enforcement — the speed bump
def enforce_refund_limit(tool_name, tool_input):
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 500:
        return {"blocked": True, "reason": "Refund exceeds $500 limit."}
    return {"blocked": False}

# Demonstration
print("Prompt approach: 'Please do not exceed $500 refunds'")
print(f"  Model processes $750 refund? Sometimes yes, sometimes no.")
print()
print("Hook approach: enforce_refund_limit('process_refund', {'amount': 750})")
result = enforce_refund_limit("process_refund", {"amount": 750})
print(f"  Result: {result}")
print(f"  Blocked 100% of the time? {result['blocked']}")
print()
print("Key insight: hooks are deterministic, prompts are probabilistic.")

In [ ]:
#@title 🎧 Listen: Mock Sdk
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_mock_sdk.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Core Implementation — Mock Agent SDK with Hooks

In [ ]:
@dataclass
class ToolUseBlock:
    type: str = "tool_use"
    id: str = field(default_factory=lambda: f"toolu_{uuid.uuid4().hex[:12]}")
    name: str = ""
    input: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TextBlock:
    type: str = "text"
    text: str = ""

@dataclass
class MockResponse:
    content: list = field(default_factory=list)
    stop_reason: str = "end_turn"
    model: str = "claude-sonnet-4-20250514"

# Databases
ORDER_DB = {
    "ORD-1234": {"status": "shipped", "tracking": "TRK-5678", "item": "Wireless Headphones",
                 "amount": 79.99, "timestamp": 1710000000},
    "ORD-5555": {"status": "processing", "tracking": None, "item": "USB-C Cable",
                 "amount": 12.99, "timestamp": 1710100000},
    "ORD-9999": {"status": "delivered", "tracking": "TRK-1111", "item": "Laptop Stand",
                 "amount": 45.00, "timestamp": 1709500000},
    "ORD-7777": {"status": "delivered", "tracking": "TRK-2222", "item": "4K Monitor",
                 "amount": 649.99, "timestamp": 1709800000},
}

ACCOUNT_DB = {
    "ACCT-100": {"name": "Alice Smith", "tier": "premium", "balance": 1200.00,
                 "created_at": 1672531200},
    "ACCT-200": {"name": "Bob Jones", "tier": "standard", "balance": 340.50,
                 "created_at": 1680307200},
}

def execute_tool(tool_name, tool_input):
    if tool_name == "lookup_order":
        oid = tool_input.get("order_id", "")
        return json.dumps(ORDER_DB.get(oid, {"error": f"Not found: {oid}"}))
    elif tool_name == "lookup_account":
        aid = tool_input.get("account_id", "")
        return json.dumps(ACCOUNT_DB.get(aid, {"error": f"Not found: {aid}"}))
    elif tool_name == "process_refund":
        return json.dumps({"success": True, "refund_id": f"REF-{uuid.uuid4().hex[:6]}",
                           "amount": tool_input.get("amount", 0)})
    elif tool_name == "update_billing":
        return json.dumps({"success": True, "account_id": tool_input.get("account_id", ""),
                           "new_email": tool_input.get("email", "")})
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

TOOLS = [
    {"name": "lookup_order", "description": "Look up order by ID",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}},
    {"name": "lookup_account", "description": "Look up customer account",
     "input_schema": {"type": "object", "properties": {"account_id": {"type": "string"}}, "required": ["account_id"]}},
    {"name": "process_refund", "description": "Process a refund",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}}, "required": ["order_id", "amount"]}},
    {"name": "update_billing", "description": "Update billing email",
     "input_schema": {"type": "object", "properties": {"account_id": {"type": "string"}, "email": {"type": "string"}}, "required": ["account_id", "email"]}},
]

print(f"Tools available: {[t['name'] for t in TOOLS]}")
print(f"Orders: {list(ORDER_DB.keys())}")
print(f"Accounts: {list(ACCOUNT_DB.keys())}")

In [ ]:
class MockAgentSDK:
    """Mock Agent SDK with PreToolUse and PostToolUse hook support."""

    def __init__(self):
        self.call_count = 0
        self.scenario = []
        self.pre_tool_hooks: List[Callable] = []
        self.post_tool_hooks: List[Callable] = []
        self.hook_log: List[Dict] = []

    def register_pre_tool_hook(self, hook: Callable):
        """Register a hook that fires BEFORE tool execution."""
        self.pre_tool_hooks.append(hook)
        print(f"  Registered PreToolUse hook: {hook.__name__}")

    def register_post_tool_hook(self, hook: Callable):
        """Register a hook that fires AFTER tool execution."""
        self.post_tool_hooks.append(hook)
        print(f"  Registered PostToolUse hook: {hook.__name__}")

    def set_scenario(self, responses):
        self.scenario = responses
        self.call_count = 0

    def create_message(self, model, max_tokens, tools, messages):
        if self.call_count < len(self.scenario):
            r = self.scenario[self.call_count]
            self.call_count += 1
            return r
        return MockResponse(content=[TextBlock(text="Done.")], stop_reason="end_turn")

    def run_pre_hooks(self, tool_name, tool_input):
        """Run all PreToolUse hooks. Returns block result if any hook blocks."""
        for hook in self.pre_tool_hooks:
            result = hook(tool_name, tool_input)
            self.hook_log.append({
                "phase": "pre", "hook": hook.__name__,
                "tool": tool_name, "result": result
            })
            if result.get("blocked"):
                return result
        return {"blocked": False}

    def run_post_hooks(self, tool_name, tool_result_str):
        """Run all PostToolUse hooks. Each can transform the result."""
        result_data = json.loads(tool_result_str)
        for hook in self.post_tool_hooks:
            result_data = hook(tool_name, result_data)
            self.hook_log.append({
                "phase": "post", "hook": hook.__name__,
                "tool": tool_name, "result_preview": str(result_data)[:80]
            })
        return json.dumps(result_data)

sdk = MockAgentSDK()
print("MockAgentSDK created with hook support!")
print(f"  Pre-hook slots: {len(sdk.pre_tool_hooks)}")
print(f"  Post-hook slots: {len(sdk.post_tool_hooks)}")

In [ ]:
#@title 🎧 Listen: Post Tool Hook
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_post_tool_hook.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. PostToolUse Hook — Data Normalization

The first hook pattern normalizes data before the model sees it. Our order database returns Unix timestamps, but we want the model to always receive ISO 8601 formatted dates.

In [ ]:
def normalize_timestamps(tool_name, tool_result):
    """PostToolUse hook: convert Unix timestamps to ISO 8601."""
    fields_to_convert = ["timestamp", "created_at", "updated_at", "shipped_at"]
    for field_name in fields_to_convert:
        if field_name in tool_result:
            unix_ts = tool_result[field_name]
            if isinstance(unix_ts, (int, float)):
                dt = datetime.fromtimestamp(unix_ts, tz=timezone.utc)
                tool_result[field_name] = dt.isoformat()
    return tool_result

# Test the hook standalone
raw_order = json.loads(execute_tool("lookup_order", {"order_id": "ORD-1234"}))
print("BEFORE hook:")
print(f"  timestamp: {raw_order['timestamp']} (Unix epoch)")
print()

normalized = normalize_timestamps("lookup_order", raw_order.copy())
print("AFTER hook:")
print(f"  timestamp: {normalized['timestamp']} (ISO 8601)")
print()

# Verify the conversion math
unix_ts = 1710000000
dt = datetime.fromtimestamp(unix_ts, tz=timezone.utc)
print(f"Conversion check: {unix_ts} seconds since epoch")
print(f"  = {unix_ts / 86400:.2f} days after Jan 1, 1970")
print(f"  = {dt.strftime('%B %d, %Y at %H:%M:%S UTC')}")
print(f"  = {dt.isoformat()}")

In [ ]:
# Register the hook and run the agentic loop WITH normalization

def run_loop_with_hooks(sdk, tools, user_message, max_turns=10, verbose=True):
    """Agentic loop that runs PreToolUse and PostToolUse hooks."""
    messages = [{"role": "user", "content": user_message}]
    sdk.hook_log = []
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = sdk.create_message(
            model="claude-sonnet-4-20250514", max_tokens=1024,
            tools=tools, messages=messages
        )

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            # PreToolUse hooks
            pre_result = sdk.run_pre_hooks(tool_block.name, tool_block.input)
            if pre_result.get("blocked"):
                if verbose:
                    print(f"  BLOCKED by hook: {pre_result['reason']}")
                result = json.dumps(pre_result)
            else:
                # Execute the tool
                result = execute_tool(tool_block.name, tool_block.input)
                if verbose:
                    print(f"  Raw result: {result}")
                # PostToolUse hooks
                result = sdk.run_post_hooks(tool_block.name, result)
                if verbose:
                    print(f"  After hooks: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [
                {"type": "tool_result", "tool_use_id": tool_block.id, "content": result}
            ]})

    return "Max turns reached"

# Register the timestamp normalization hook
print("Registering hooks...")
sdk.register_post_tool_hook(normalize_timestamps)

# Run a scenario — the model looks up an order
sdk.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})],
                 stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Order ORD-1234: Wireless Headphones shipped on 2024-03-09T16:00:00+00:00 via tracking TRK-5678.")],
                 stop_reason="end_turn"),
])

run_loop_with_hooks(sdk, TOOLS, "What is the status of order ORD-1234?")

Notice how the model received `"2024-03-09T16:00:00+00:00"` instead of the raw Unix epoch `1710000000`. The hook transformed the data silently — the model never had to parse a raw integer timestamp. This is **transparent normalization**: the hook pipeline runs between tool execution and model consumption, ensuring data consistency without any prompt engineering.

In [ ]:
#@title 🎧 Listen: Pre Tool Hook
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_pre_tool_hook.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. PreToolUse Hook — Policy Enforcement

Now let us build the enforcement side. We want to block any refund over $500 and any billing update to accounts in the "standard" tier without supervisor approval.

In [ ]:
def enforce_refund_policy(tool_name, tool_input):
    """PreToolUse hook: block refunds exceeding $500."""
    if tool_name == "process_refund":
        amount = tool_input.get("amount", 0)
        if amount > 500:
            return {
                "blocked": True,
                "reason": f"Refund of ${amount:.2f} exceeds $500 limit. "
                          f"Manager approval required. "
                          f"Escalate to supervisor with order details."
            }
    return {"blocked": False}

def enforce_billing_tier(tool_name, tool_input):
    """PreToolUse hook: block billing changes for non-premium accounts."""
    if tool_name == "update_billing":
        account_id = tool_input.get("account_id", "")
        account = ACCOUNT_DB.get(account_id, {})
        if account.get("tier") != "premium":
            return {
                "blocked": True,
                "reason": f"Account {account_id} is '{account.get('tier', 'unknown')}' tier. "
                          f"Only premium accounts can self-service billing changes."
            }
    return {"blocked": False}

# Register enforcement hooks
sdk2 = MockAgentSDK()
sdk2.register_post_tool_hook(normalize_timestamps)
sdk2.register_pre_tool_hook(enforce_refund_policy)
sdk2.register_pre_tool_hook(enforce_billing_tier)

print(f"\nHook pipeline: {len(sdk2.pre_tool_hooks)} pre-hooks, {len(sdk2.post_tool_hooks)} post-hooks")

# Test 1: Refund under $500 — should pass
print("\n" + "="*50)
print("Test 1: Refund $45.00 (under limit)")
r1 = sdk2.run_pre_hooks("process_refund", {"order_id": "ORD-9999", "amount": 45.00})
print(f"  Blocked: {r1['blocked']}")

# Test 2: Refund over $500 — should block
print("\nTest 2: Refund $649.99 (over limit)")
r2 = sdk2.run_pre_hooks("process_refund", {"order_id": "ORD-7777", "amount": 649.99})
print(f"  Blocked: {r2['blocked']}")
print(f"  Reason: {r2['reason']}")

# Test 3: Billing update for premium — should pass
print("\nTest 3: Billing update for ACCT-100 (premium)")
r3 = sdk2.run_pre_hooks("update_billing", {"account_id": "ACCT-100", "email": "alice@new.com"})
print(f"  Blocked: {r3['blocked']}")

# Test 4: Billing update for standard — should block
print("\nTest 4: Billing update for ACCT-200 (standard)")
r4 = sdk2.run_pre_hooks("update_billing", {"account_id": "ACCT-200", "email": "bob@new.com"})
print(f"  Blocked: {r4['blocked']}")
print(f"  Reason: {r4['reason']}")

In [ ]:
# Full scenario: Agent tries to refund a $649.99 monitor — hook blocks it

sdk2.set_scenario([
    MockResponse(
        content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-7777"})],
        stop_reason="tool_use"
    ),
    MockResponse(
        content=[ToolUseBlock(name="process_refund", input={"order_id": "ORD-7777", "amount": 649.99})],
        stop_reason="tool_use"
    ),
    MockResponse(
        content=[TextBlock(text="I attempted to process your refund of $649.99 for the 4K Monitor, but it was blocked because it exceeds our $500 automatic refund limit. I have escalated this to a manager for approval. You should hear back within 24 hours.")],
        stop_reason="end_turn"
    ),
])

run_loop_with_hooks(sdk2, TOOLS, "I need a refund for order ORD-7777")

The agent tried to process a $649.99 refund and the PreToolUse hook blocked it deterministically. The model then received the block reason as the tool result and adapted its response — telling the customer about the escalation. This is the pattern: **hooks enforce, the model adapts.**

In [ ]:
#@title 🎧 Listen: Exercise1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_exercise1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Exercise 1 — Build a Data Sanitization Hook

In [ ]:
# ============ TODO ============
# Build a PostToolUse hook that redacts sensitive data before the model sees it.
# Requirements:
#   1. If the tool result contains a "balance" field, replace the exact value
#      with a range: "low" (<100), "medium" (100-999), "high" (1000+)
#   2. If the tool result contains a "name" field, redact the last name
#      (keep only the first name + "***")
#   3. Return the modified result dict
#
# Example:
#   Input:  {"name": "Alice Smith", "balance": 1200.00, "tier": "premium"}
#   Output: {"name": "Alice ***", "balance": "high", "tier": "premium"}

def sanitize_pii(tool_name, tool_result):
    # YOUR CODE HERE
    pass

# Test it:
# test_data = {"name": "Alice Smith", "balance": 1200.00, "tier": "premium"}
# print(sanitize_pii("lookup_account", test_data))
# Expected: {"name": "Alice ***", "balance": "high", "tier": "premium"}
# ==============================

### Exercise 1 — Solution

In [ ]:
def sanitize_pii(tool_name, tool_result):
    """PostToolUse hook: redact PII from tool results."""
    # Redact last name
    if "name" in tool_result and isinstance(tool_result["name"], str):
        parts = tool_result["name"].split()
        if len(parts) > 1:
            tool_result["name"] = parts[0] + " ***"

    # Replace balance with range
    if "balance" in tool_result and isinstance(tool_result["balance"], (int, float)):
        bal = tool_result["balance"]
        if bal < 100:
            tool_result["balance"] = "low"
        elif bal < 1000:
            tool_result["balance"] = "medium"
        else:
            tool_result["balance"] = "high"

    return tool_result

# Test
test_cases = [
    {"name": "Alice Smith", "balance": 1200.00, "tier": "premium"},
    {"name": "Bob Jones", "balance": 340.50, "tier": "standard"},
    {"name": "Carol", "balance": 50.00, "tier": "basic"},
]

for tc in test_cases:
    original = tc.copy()
    sanitized = sanitize_pii("lookup_account", tc)
    print(f"  {original} -> {sanitized}")

In [ ]:
#@title 🎧 Listen: Exercise2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_exercise2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 2 — Build a Rate Limiting Hook

In [ ]:
# ============ TODO ============
# Build a PreToolUse hook that enforces rate limits per tool.
# Requirements:
#   1. Track how many times each tool has been called (use a dict)
#   2. If "process_refund" is called more than 3 times, block it
#      with reason "Rate limit exceeded: max 3 refunds per session"
#   3. If "update_billing" is called more than 2 times, block it
#      with reason "Rate limit exceeded: max 2 billing updates per session"
#   4. All other tools have no limit
#
# Hint: You will need a mutable object (dict) defined outside the function
#       to track call counts across invocations.

call_counts = {}

def enforce_rate_limits(tool_name, tool_input):
    # YOUR CODE HERE
    pass

# Test it:
# for i in range(5):
#     result = enforce_rate_limits("process_refund", {"order_id": "ORD-1234", "amount": 10})
#     print(f"  Call {i+1}: blocked={result.get('blocked', False)}")
# Expected: calls 1-3 pass, calls 4-5 blocked
# ==============================

### Exercise 2 — Solution

In [ ]:
call_counts = {}
RATE_LIMITS = {
    "process_refund": 3,
    "update_billing": 2,
}

def enforce_rate_limits(tool_name, tool_input):
    """PreToolUse hook: enforce per-tool rate limits."""
    if tool_name in RATE_LIMITS:
        call_counts[tool_name] = call_counts.get(tool_name, 0) + 1
        limit = RATE_LIMITS[tool_name]
        if call_counts[tool_name] > limit:
            return {
                "blocked": True,
                "reason": f"Rate limit exceeded: max {limit} "
                          f"{tool_name.replace('_', ' ')}s per session"
            }
    return {"blocked": False}

# Test process_refund (limit = 3)
print("Testing process_refund rate limit (max 3):")
call_counts.clear()
for i in range(5):
    result = enforce_rate_limits("process_refund", {"order_id": "ORD-1234", "amount": 10})
    status = f"BLOCKED: {result['reason']}" if result.get("blocked") else "allowed"
    print(f"  Call {i+1}: {status}")

# Test update_billing (limit = 2)
print("\nTesting update_billing rate limit (max 2):")
call_counts.clear()
for i in range(4):
    result = enforce_rate_limits("update_billing", {"account_id": "ACCT-100", "email": "x@y.com"})
    status = f"BLOCKED: {result['reason']}" if result.get("blocked") else "allowed"
    print(f"  Call {i+1}: {status}")

# Test lookup_order (no limit)
print("\nTesting lookup_order (no limit):")
call_counts.clear()
for i in range(5):
    result = enforce_rate_limits("lookup_order", {"order_id": "ORD-1234"})
    print(f"  Call {i+1}: allowed={not result.get('blocked', False)}")

In [ ]:
#@title 🎧 Listen: Exercise3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_exercise3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 3 — Hooks vs Prompts Comparison

In [ ]:
# ============ TODO ============
# This exercise demonstrates WHY hooks beat prompts for compliance.
# You will simulate 100 "model decisions" to see the reliability difference.
#
# 1. Write a function `prompt_based_check(amount)` that simulates a prompt
#    telling the model "do not refund over $500". The model "obeys" 95% of
#    the time — use `random.random() < 0.95` to simulate this.
#    Return True if refund is allowed, False if blocked.
#
# 2. Write a function `hook_based_check(amount)` that deterministically
#    blocks any refund over $500. No randomness.
#    Return True if refund is allowed, False if blocked.
#
# 3. Run both functions 100 times with amount=750 and count how many
#    times each one INCORRECTLY allows the refund.
#
# The result should clearly show: hooks = 0 failures, prompts = ~5 failures.

import random
random.seed(42)

def prompt_based_check(amount):
    # YOUR CODE HERE
    pass

def hook_based_check(amount):
    # YOUR CODE HERE
    pass

# Run the comparison
# prompt_failures = sum(1 for _ in range(100) if prompt_based_check(750))
# hook_failures = sum(1 for _ in range(100) if hook_based_check(750))
# print(f"Prompt-based: {prompt_failures}/100 incorrectly allowed")
# print(f"Hook-based:   {hook_failures}/100 incorrectly allowed")
# ==============================

### Exercise 3 — Solution

In [ ]:
import random
random.seed(42)

def prompt_based_check(amount):
    """Simulate prompt-based enforcement — 95% compliance rate."""
    if amount > 500:
        # Model "obeys" the prompt 95% of the time
        return random.random() >= 0.95  # True = allowed (bad), False = blocked (good)
    return True  # Under limit, always allow

def hook_based_check(amount):
    """Hook-based enforcement — 100% compliance rate."""
    if amount > 500:
        return False  # Always blocked, no exceptions
    return True

# Run 100 simulations with a $750 refund
prompt_failures = sum(1 for _ in range(100) if prompt_based_check(750))
hook_failures = sum(1 for _ in range(100) if hook_based_check(750))

print("Simulating 100 attempts to refund $750 (over $500 limit):")
print(f"  Prompt-based: {prompt_failures}/100 incorrectly allowed")
print(f"  Hook-based:   {hook_failures}/100 incorrectly allowed")
print()
print(f"Prompt reliability: {100 - prompt_failures}%")
print(f"Hook reliability:   {100 - hook_failures}%")
print()
if prompt_failures > 0:
    print(f"In a system processing 10,000 refunds/day, prompt-based enforcement")
    print(f"would allow ~{prompt_failures * 100} unauthorized refunds daily.")
    print(f"Hook-based enforcement: 0 unauthorized refunds. Ever.")
print()
print("Exam takeaway: hooks for HARD rules, prompts for SOFT guidance.")

In [ ]:
#@title 🎧 Listen: Exercise4
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_exercise4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 4 — Build a Complete Hook Pipeline

In [ ]:
# ============ TODO ============
# Wire together a complete hook pipeline with these hooks:
#   PreToolUse:
#     1. enforce_refund_policy (block refunds > $500) — already defined above
#     2. enforce_business_hours (block process_refund outside 9am-5pm UTC)
#   PostToolUse:
#     1. normalize_timestamps — already defined above
#     2. add_audit_trail (add an "audited_at" ISO timestamp and "audited_by": "hook_pipeline")
#
# Instructions:
#   a) Write enforce_business_hours(tool_name, tool_input):
#      - Only applies to "process_refund"
#      - Use a hardcoded simulated hour (e.g., current_hour = 22 for 10pm)
#      - Block if hour < 9 or hour >= 17
#   b) Write add_audit_trail(tool_name, tool_result):
#      - Add "audited_at" with current ISO timestamp
#      - Add "audited_by" with value "hook_pipeline"
#   c) Create a new MockAgentSDK, register all 4 hooks, and run the
#      scenario below to verify the pipeline works.

def enforce_business_hours(tool_name, tool_input):
    # YOUR CODE HERE (simulate with current_hour = 22)
    pass

def add_audit_trail(tool_name, tool_result):
    # YOUR CODE HERE
    pass

# Create sdk3, register hooks, then run:
# sdk3.set_scenario([
#     MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})],
#                  stop_reason="tool_use"),
#     MockResponse(content=[TextBlock(text="Order details retrieved with audit trail.")],
#                  stop_reason="end_turn"),
# ])
# run_loop_with_hooks(sdk3, TOOLS, "Look up ORD-1234")
# ==============================

### Exercise 4 — Solution

In [ ]:
def enforce_business_hours(tool_name, tool_input):
    """PreToolUse hook: block refunds outside business hours (9am-5pm UTC)."""
    if tool_name == "process_refund":
        current_hour = 22  # Simulated: 10pm UTC
        if current_hour < 9 or current_hour >= 17:
            return {
                "blocked": True,
                "reason": f"Refunds cannot be processed outside business hours "
                          f"(9am-5pm UTC). Current time: {current_hour}:00 UTC."
            }
    return {"blocked": False}

def add_audit_trail(tool_name, tool_result):
    """PostToolUse hook: add audit metadata to every tool result."""
    tool_result["audited_at"] = datetime.now(timezone.utc).isoformat()
    tool_result["audited_by"] = "hook_pipeline"
    return tool_result

# Create fresh SDK and register all hooks
sdk3 = MockAgentSDK()
print("Registering complete hook pipeline...")
sdk3.register_pre_tool_hook(enforce_refund_policy)
sdk3.register_pre_tool_hook(enforce_business_hours)
sdk3.register_post_tool_hook(normalize_timestamps)
sdk3.register_post_tool_hook(add_audit_trail)
print(f"Pipeline: {len(sdk3.pre_tool_hooks)} pre-hooks, {len(sdk3.post_tool_hooks)} post-hooks")

# Test 1: Lookup order — should pass pre-hooks and get normalized + audited
print("\n" + "="*50)
print("Test 1: lookup_order (no restrictions)")
sdk3.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})],
                 stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Order ORD-1234 retrieved with full audit trail.")],
                 stop_reason="end_turn"),
])
run_loop_with_hooks(sdk3, TOOLS, "Look up ORD-1234")

# Test 2: Refund at 10pm — should be blocked by business hours
print("\n" + "="*50)
print("Test 2: process_refund at 10pm UTC (blocked by business hours)")
sdk3.set_scenario([
    MockResponse(content=[ToolUseBlock(name="process_refund", input={"order_id": "ORD-9999", "amount": 45.00})],
                 stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Refund blocked — outside business hours.")],
                 stop_reason="end_turn"),
])
run_loop_with_hooks(sdk3, TOOLS, "Refund order ORD-9999 ($45)")

In [ ]:
#@title 🎧 Listen: Exercise5
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_exercise5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 5 — Hook Ordering Matters

In [ ]:
# ============ TODO ============
# Demonstrate that hook ordering matters by answering these questions
# in the comments, then verify with code.
#
# Scenario: You have two PostToolUse hooks registered in this order:
#   1. normalize_timestamps (converts Unix ints to ISO strings)
#   2. redact_old_dates (replaces any timestamp before 2024 with "REDACTED")
#
# Question A: If redact_old_dates checks `isinstance(value, int)` to detect
#             Unix timestamps, will it work correctly when registered AFTER
#             normalize_timestamps? Why or why not?
#
# Question B: What would happen if you reversed the order — registering
#             redact_old_dates FIRST, then normalize_timestamps?
#
# Instructions:
#   1. Write redact_old_dates(tool_name, tool_result) that checks the
#      "timestamp" field. If it is a string containing a year before 2024,
#      replace it with "REDACTED". If it is an int < 1704067200 (Jan 1 2024),
#      replace it with "REDACTED".
#   2. Test with both orderings and print the results.

def redact_old_dates(tool_name, tool_result):
    # YOUR CODE HERE
    pass

# Test both orderings with ORD-9999 (timestamp: 1709500000 = March 3, 2024)
# and a custom order with timestamp 1672531200 (Jan 1, 2023)
# ==============================

### Exercise 5 — Solution

In [ ]:
def redact_old_dates(tool_name, tool_result):
    """PostToolUse hook: redact timestamps from before 2024."""
    if "timestamp" in tool_result:
        ts = tool_result["timestamp"]
        if isinstance(ts, (int, float)):
            # Unix epoch: Jan 1, 2024 = 1704067200
            if ts < 1704067200:
                tool_result["timestamp"] = "REDACTED"
        elif isinstance(ts, str):
            # ISO string: check if year < 2024
            try:
                year = int(ts[:4])
                if year < 2024:
                    tool_result["timestamp"] = "REDACTED"
            except (ValueError, IndexError):
                pass
    return tool_result

# Order 1: normalize_timestamps THEN redact_old_dates
print("ORDER 1: normalize_timestamps → redact_old_dates")
test1 = {"order_id": "ORD-old", "timestamp": 1672531200}  # Jan 1, 2023
result1 = normalize_timestamps("test", test1.copy())
print(f"  After normalize: timestamp = {result1['timestamp']}")
result1 = redact_old_dates("test", result1)
print(f"  After redact:    timestamp = {result1['timestamp']}")

print()

# Order 2: redact_old_dates THEN normalize_timestamps
print("ORDER 2: redact_old_dates → normalize_timestamps")
test2 = {"order_id": "ORD-old", "timestamp": 1672531200}  # Jan 1, 2023
result2 = redact_old_dates("test", test2.copy())
print(f"  After redact:    timestamp = {result2['timestamp']}")
result2 = normalize_timestamps("test", result2)
print(f"  After normalize: timestamp = {result2['timestamp']}")

print()
print("Analysis:")
print("  Order 1: normalize converts int→ISO string, redact checks string year → works correctly")
print("  Order 2: redact checks int < threshold → replaces with 'REDACTED' string,")
print("           normalize sees a string (not int) → skips it → works correctly")
print()
print("  But consider a 2024 timestamp (should NOT be redacted):")
test3 = {"order_id": "ORD-new", "timestamp": 1710000000}  # March 2024
print(f"\n  Order 1 with timestamp={test3['timestamp']} (March 2024):")
r3 = normalize_timestamps("test", test3.copy())
r3 = redact_old_dates("test", r3)
print(f"    Final: {r3['timestamp']} (correct — not redacted)")

print()
print("Key insight: Hook ordering creates a pipeline. Each hook sees")
print("the OUTPUT of the previous hook, not the original data.")

In [ ]:
#@title 🎧 Listen: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Visualization — Hook Pipeline Flow

In [ ]:
def visualize_hook_pipeline(sdk_instance):
    """Visualize the complete hook pipeline and execution log."""
    print("=" * 65)
    print("          HOOK PIPELINE ARCHITECTURE")
    print("=" * 65)
    print()

    # Show registered hooks
    print("  PreToolUse Hooks (fire BEFORE tool execution):")
    if sdk_instance.pre_tool_hooks:
        for i, hook in enumerate(sdk_instance.pre_tool_hooks, 1):
            print(f"    [{i}] {hook.__name__}")
    else:
        print("    (none)")

    print()
    print("  PostToolUse Hooks (fire AFTER tool execution):")
    if sdk_instance.post_tool_hooks:
        for i, hook in enumerate(sdk_instance.post_tool_hooks, 1):
            print(f"    [{i}] {hook.__name__}")
    else:
        print("    (none)")

    print()
    print("  Execution Flow:")
    print("  ┌─────────────────┐")
    print("  │  Model requests │")
    print("  │   tool call     │")
    print("  └────────┬────────┘")
    print("           │")
    print("           ▼")
    print("  ┌─────────────────┐")
    print("  │  PreToolUse     │──── Can BLOCK the call")
    for hook in sdk_instance.pre_tool_hooks:
        print(f"  │  • {hook.__name__[:15]:<15}│")
    print("  └────────┬────────┘")
    print("           │ (if not blocked)")
    print("           ▼")
    print("  ┌─────────────────┐")
    print("  │  Execute Tool   │")
    print("  │  (get raw data) │")
    print("  └────────┬────────┘")
    print("           │")
    print("           ▼")
    print("  ┌─────────────────┐")
    print("  │  PostToolUse    │──── Can TRANSFORM the result")
    for hook in sdk_instance.post_tool_hooks:
        print(f"  │  • {hook.__name__[:15]:<15}│")
    print("  └────────┬────────┘")
    print("           │")
    print("           ▼")
    print("  ┌─────────────────┐")
    print("  │  Model receives │")
    print("  │  clean result   │")
    print("  └─────────────────┘")

    # Show hook log if available
    if sdk_instance.hook_log:
        print()
        print("  Recent Hook Log:")
        for entry in sdk_instance.hook_log[-6:]:
            phase = "PRE " if entry["phase"] == "pre" else "POST"
            print(f"    [{phase}] {entry['hook']} on {entry['tool']}")

visualize_hook_pipeline(sdk3)

In [ ]:
#@title 🎧 Listen: Mini Project
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_mini_project.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Mini-Project — Financial Operations Agent with Full Hook Pipeline

Build a complete financial operations agent that processes customer requests with a full hook pipeline ensuring compliance.

In [ ]:
# Mini-Project: Financial Operations Agent
# This agent handles order lookups, account queries, refunds, and billing updates.
# The hook pipeline ensures:
#   - No refunds > $500 (PreToolUse)
#   - No refunds outside 9am-5pm UTC (PreToolUse)
#   - All timestamps normalized to ISO 8601 (PostToolUse)
#   - All results include audit trail (PostToolUse)
#   - PII is redacted before the model sees it (PostToolUse)

class FinancialOpsAgent:
    """Complete financial operations agent with hook-based compliance."""

    def __init__(self):
        self.sdk = MockAgentSDK()
        self._register_hooks()
        self.audit_log = []

    def _register_hooks(self):
        print("Initializing Financial Operations Agent...")

        # PreToolUse hooks — enforcement
        self.sdk.register_pre_tool_hook(enforce_refund_policy)

        # PostToolUse hooks — normalization and auditing
        self.sdk.register_post_tool_hook(normalize_timestamps)
        self.sdk.register_post_tool_hook(self._log_for_audit)

        print(f"Agent ready with {len(self.sdk.pre_tool_hooks)} enforcement hooks "
              f"and {len(self.sdk.post_tool_hooks)} normalization hooks.\n")

    def _log_for_audit(self, tool_name, tool_result):
        """PostToolUse hook: log every tool result for compliance audit."""
        self.audit_log.append({
            "tool": tool_name,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "result_keys": list(tool_result.keys()) if isinstance(tool_result, dict) else "non-dict",
        })
        return tool_result

    def handle_request(self, scenario, user_message):
        """Handle a customer request with full hook pipeline."""
        self.sdk.set_scenario(scenario)
        result = run_loop_with_hooks(self.sdk, TOOLS, user_message)
        return result

    def print_audit_log(self):
        print(f"\n{'='*50}")
        print(f"AUDIT LOG ({len(self.audit_log)} entries)")
        print(f"{'='*50}")
        for i, entry in enumerate(self.audit_log, 1):
            print(f"  [{i}] {entry['tool']} at {entry['timestamp']}")
            print(f"      Result keys: {entry['result_keys']}")

# Create the agent
agent = FinancialOpsAgent()

# Scenario 1: Simple order lookup
print("SCENARIO 1: Order Lookup")
agent.handle_request(
    scenario=[
        MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})],
                     stop_reason="tool_use"),
        MockResponse(content=[TextBlock(text="Order ORD-1234: Wireless Headphones, shipped, tracking TRK-5678.")],
                     stop_reason="end_turn"),
    ],
    user_message="What is the status of order ORD-1234?"
)

# Scenario 2: Refund blocked by policy
print("\n\nSCENARIO 2: Refund Blocked")
agent.handle_request(
    scenario=[
        MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-7777"})],
                     stop_reason="tool_use"),
        MockResponse(content=[ToolUseBlock(name="process_refund", input={"order_id": "ORD-7777", "amount": 649.99})],
                     stop_reason="tool_use"),
        MockResponse(content=[TextBlock(text="Your $649.99 refund requires manager approval. Escalating now.")],
                     stop_reason="end_turn"),
    ],
    user_message="Refund my 4K Monitor order ORD-7777"
)

# Scenario 3: Small refund allowed
print("\n\nSCENARIO 3: Refund Allowed")
agent.handle_request(
    scenario=[
        MockResponse(content=[ToolUseBlock(name="process_refund", input={"order_id": "ORD-5555", "amount": 12.99})],
                     stop_reason="tool_use"),
        MockResponse(content=[TextBlock(text="Refund of $12.99 for USB-C Cable processed successfully.")],
                     stop_reason="end_turn"),
    ],
    user_message="Refund ORD-5555"
)

# Show audit trail
agent.print_audit_log()

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Key Takeaways

**Exam-critical concepts for Task Statement 1.5 (Hooks & Compliance):**

- **PostToolUse hooks** transform tool results before the model sees them — use for data normalization (Unix timestamps to ISO 8601, PII redaction, unit conversion)
- **PreToolUse hooks** intercept tool calls before execution and can block them — use for policy enforcement (refund limits, rate limiting, business hours)
- **Hooks are deterministic; prompts are probabilistic.** For hard compliance requirements (financial limits, data access controls, regulatory rules), always use hooks
- **Prompts are for soft guidance** — tone, style, clarifying questions, best-effort preferences
- **Hook ordering matters** — each hook in the pipeline receives the output of the previous hook, not the original data
- **The model adapts to hook results** — when a hook blocks a tool call, the block reason becomes the tool result, and the model uses it to inform the user

In [ ]:
print("Notebook 3 complete!")
print()
print("Key exam formula:")
print("  Hard compliance requirement → Hook (deterministic)")
print("  Soft guidance preference   → Prompt (probabilistic)")
print()
print("Next: Task Decomposition Strategies (NB 4)")